# 用 Claude API 实现程序化工具使用（PTC）

程序化工具使用（Programmatic Tool Calling PTC）让 Claude 能够编写在代码执行环境（Code Execution environment）内以编程方式调用工具的代码，而不必每次工具使用都在模型与调用方之间往返。这大幅降低了多次工具使用的端到端时延，并且由于模型可以编写代码、在数据进入模型的上下文窗口之前就剔除无关上下文（例如在庞大而嘈杂的文件中用 grep 提取关键信息），词元消耗也能显著减少。

当你面对可能无法直接修改的第三方 API 和工具时，PTC 可以通过让 Claude 编写可在代码执行环境中调用的代码，帮助减少上下文占用。

在本 cookbook 中，我们将使用一个团队费用管理的模拟 API。该 API 被设计为需要多次调用，并会返回大量结果，以此帮助说明程序化工具使用的收益。

## 完成本 cookbook 后，你将能够：

- 理解常规工具使用与程序化工具使用（PTC）之间的区别
- 编写利用 PTC 的智能体


## 前置条件

在跟随本指南之前，请确认你已具备：

**必备知识**

- Python 基础——熟悉 async/await、函数和基本数据结构
- 对智能体模式和工具使用有基本理解

**必备工具**

- Python 3.11 或更高版本
- Anthropic API 密钥
- Anthropic Python SDK >= 0.72


## 设置

首先，安装所需依赖：

In [ ]:
# %pip install -r requirements.txt

说明：确认你的 .env 文件包含：

`ANTHROPIC_API_KEY=your_key_here`

加载环境变量并配置客户端。我们还会加载一个用于可视化 Claude 消息响应的辅助工具。


In [2]:
from dotenv import load_dotenv
from utils.visualize import visualize

load_dotenv()

MODEL = "claude-sonnet-4-6"

viz = visualize(auto_show=True)

## 理解第三方 API

在 [utils/team_expense_api.py](utils/team_expense_api.py) 中定义了三个函数：`get_team_members`、`get_expenses` 和 `get_custom_budget`。`get_team_members` 函数让我们可以获取某个部门的全部员工及其角色、级别和联系信息。`get_expenses` 函数返回某位员工在特定季度的全部费用明细——每位员工可能有数百条记录，每条记录都带有大量元数据，包括收据 URL、审批链、商户信息等。`get_custom_budget` 函数检查特定员工是否享有自定义差旅预算例外（否则该员工使用标准的 $5,000 季度限额）。

在这个场景中，我们需要分析团队费用，并找出哪些员工超出了预算。传统做法可能是逐人手动导出费用报表、按类别汇总费用、对照预算限额比较（并检查自定义预算例外），再编制一份报告。这里我们改为让 Claude 代我们完成这项分析：用可用的工具获取团队数据、拉取可能多达数百条、元数据丰富的费用明细，并判断谁超出了预算。

这里的关键挑战是：每位员工可能有 100 多条费用明细需要获取、解析和聚合——而 `get_custom_budget` 工具只有在分析费用、确认某人超出标准预算之后才能调用。这形成了一条顺序依赖链，使它成为演示程序化工具使用收益的理想用例。

我们会把工具定义传给 messages API，并要求 Claude 执行这项分析。如果你不熟悉 Claude API 的工具使用方式，请阅读[实现工具使用](https://docs.claude.com/en/docs/agents-and-tools/tool-use/implement-tool-use)文档。

In [3]:
import json

import anthropic
from utils.team_expense_api import get_custom_budget, get_expenses, get_team_members

client = anthropic.Anthropic()

# Tool definitions for the team expense API
tools = [
    {
        "name": "get_team_members",
        "description": 'Returns a list of team members for a given department. Each team member includes their ID, name, role, level (junior, mid, senior, staff, principal), and contact information. Use this to get a list of people whose expenses you want to analyze. Available departments are: engineering, sales, and marketing.\n\nRETURN FORMAT: Returns a JSON string containing an ARRAY of team member objects (not wrapped in an outer object). Parse with json.loads() to get a list. Example: [{"id": "ENG001", "name": "Alice", ...}, {"id": "ENG002", ...}]',
        "input_schema": {
            "type": "object",
            "properties": {
                "department": {
                    "type": "string",
                    "description": "The department name. Case-insensitive.",
                }
            },
            "required": ["department"],
        },
        "input_examples": [
            {"department": "engineering"},
            {"department": "sales"},
            {"department": "marketing"},
        ],
    },
    {
        "name": "get_expenses",
        "description": "Returns all expense line items for a given employee in a specific quarter. Each expense includes extensive metadata: date, category, description, amount (in USD), currency, status (approved, pending, rejected), receipt URL, approval chain, merchant name and location, payment method, and project codes. An employee may have 20-50+ expense line items per quarter, and each line item contains substantial metadata for audit and compliance purposes. Categories include: 'travel' (flights, trains, rental cars, taxis, parking), 'lodging' (hotels, airbnb), 'meals', 'software', 'equipment', 'conference', 'office', and 'internet'. IMPORTANT: Only expenses with status='approved' should be counted toward budget limits.\n\nRETURN FORMAT: Returns a JSON string containing an ARRAY of expense objects (not wrapped in an outer object with an 'expenses' key). Parse with json.loads() to get a list directly. Example: [{\"expense_id\": \"ENG001_Q3_001\", \"amount\": 1250.50, \"category\": \"travel\", ...}, {...}]",
        "input_schema": {
            "type": "object",
            "properties": {
                "employee_id": {
                    "type": "string",
                    "description": "The unique employee identifier",
                },
                "quarter": {
                    "type": "string",
                    "description": "Quarter identifier: 'Q1', 'Q2', 'Q3', or 'Q4'",
                },
            },
            "required": ["employee_id", "quarter"],
        },
        "input_examples": [
            {"employee_id": "ENG001", "quarter": "Q3"},
            {"employee_id": "SAL002", "quarter": "Q1"},
            {"employee_id": "MKT001", "quarter": "Q4"},
        ],
    },
    {
        "name": "get_custom_budget",
        "description": 'Get the custom quarterly travel budget for a specific employee. Most employees have a standard $5,000 quarterly travel budget. However, some employees have custom budget exceptions based on their role requirements. This function checks if a specific employee has a custom budget assigned.\n\nRETURN FORMAT: Returns a JSON string containing a SINGLE OBJECT (not an array). Parse with json.loads() to get a dict. Example: {"user_id": "ENG001", "has_custom_budget": false, "travel_budget": 5000, "reason": "Standard", "currency": "USD"}',
        "input_schema": {
            "type": "object",
            "properties": {
                "user_id": {
                    "type": "string",
                    "description": "The unique employee identifier",
                }
            },
            "required": ["user_id"],
        },
        "input_examples": [
            {"user_id": "ENG001"},
            {"user_id": "SAL002"},
            {"user_id": "MKT001"},
        ],
    },
]

tool_functions = {
    "get_team_members": get_team_members,
    "get_expenses": get_expenses,
    "get_custom_budget": get_custom_budget,
}

## 传统工具使用（基线）

在第一个示例中，我们使用传统工具使用来建立基线。

我们用初始查询调用 `messages.create` API。当模型以 `tool_use` 原因停止时，我们按要求执行该工具，然后把工具输出加入消息，再次调用模型。

In [4]:
import time

from anthropic.types import TextBlock, ToolUseBlock
from anthropic.types.beta import (
    BetaMessageParam as MessageParam,
)
from anthropic.types.beta import (
    BetaTextBlock,
    BetaToolUseBlock,
)

messages: list[MessageParam] = []


def run_agent_without_ptc(user_message):
    """Run agent using traditional tool calling"""
    messages.append({"role": "user", "content": user_message})
    total_tokens = 0
    start_time = time.time()
    api_counter = 0

    while True:
        response = client.beta.messages.create(
            model=MODEL,
            max_tokens=4000,
            tools=tools,
            messages=messages,
            betas=["advanced-tool-use-2025-11-20"],
        )

        api_counter += 1

        # Track token usage
        total_tokens += response.usage.input_tokens + response.usage.output_tokens
        viz.capture(response)
        if response.stop_reason == "end_turn":
            # Extract the first text block from the response
            final_response = next(
                (
                    block.text
                    for block in response.content
                    if isinstance(block, (BetaTextBlock, TextBlock))
                ),
                None,
            )
            elapsed_time = time.time() - start_time
            return final_response, messages, total_tokens, elapsed_time, api_counter

        # Process tool calls
        if response.stop_reason == "tool_use":
            # First, add the assistant's response to messages
            messages.append({"role": "assistant", "content": response.content})

            # Collect all tool results
            tool_results = []

            for block in response.content:
                if isinstance(block, (BetaToolUseBlock, ToolUseBlock)):
                    tool_name = block.name
                    tool_input = block.input
                    tool_use_id = block.id

                    result = tool_functions[tool_name](**tool_input)

                    content = str(result)

                    tool_result = {
                        "type": "tool_result",
                        "tool_use_id": tool_use_id,
                        "content": content,
                    }
                    tool_results.append(tool_result)

            # Append all tool results at once after collecting them
            messages.append({"role": "user", "content": tool_results})

        else:
            print(f"\nUnexpected stop reason: {response.stop_reason}")
            elapsed_time = time.time() - start_time

            final_response = next(
                (
                    block.text
                    for block in response.content
                    if isinstance(block, (BetaTextBlock, TextBlock))
                ),
                f"Stopped with reason: {response.stop_reason}",
            )
            return final_response, messages, total_tokens, elapsed_time, api_counter

我们给模型的初始查询中包含一些指导模型的说明。为了简洁，我们要求模型每个工具只调用一次。如果做更深入的分析，模型可能希望查看多个系统或多个时间段。

In [5]:
query = "Which engineering team members exceeded their Q3 travel budget? Standard quarterly travel budget is $5,000. However, some employees have custom budget limits. For anyone who exceeded the $5,000 standard budget, check if they have a custom budget exception. If they do, use that custom limit instead to determine if they truly exceeded their budget."

In [6]:
# Run the agent
result, conversation, total_tokens, elapsed_time, api_count_without_ptc = run_agent_without_ptc(
    query
)

print(f"Result: {result}")
print(f"API calls made: {api_count_without_ptc}")
print(f"Total tokens used: {total_tokens:,}")
print(f"Total time taken: {elapsed_time:.2f}s")

╭────────────────────────────────────────────── Claude API Response ──────────────────────────────────────────────╮
│ Claude Message (assistant) │ tokens: 1,859 in • 85 out • 1,944 total                                            │
│ ├── Model: claude-sonnet-4-6                                                                           │
│ ├── Stop Reason: tool_use                                                                                       │
│ └── Content (2 blocks)                                                                                          │
│     ├── Block 1                                                                                                 │
│     │   └── Text                                                                                                │
│     │       └── I'll help you identify which engineering team members exceeded their Q3 travel budget. Let me   │
│     │           start by getting the list of engineering team members.                                          │
│     └── Block 2                                                                                                 │
│         └── Tool Use: get_team_members                                                                          │
│             ├── ID: toolu_01LuouuJYp1sSvBe2Du7EG7v                                                              │
│             ├── Caller: model (direct)                                                                          │
│             └── Input:                                                                                          │
│                 └── {                                                                                           │
│                       "department": "engineering"                                                               │
│                     }                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────── Claude API Response ───────────────────────────────╮
│ Claude Message (assistant) │ tokens: 2,473 in • 497 out • 2,970 total            │
│ ├── Model: claude-sonnet-4-6                                            │
│ ├── Stop Reason: tool_use                                                        │
│ └── Content (9 blocks)                                                           │
│     ├── Block 1                                                                  │
│     │   └── Text                                                                 │
│     │       └── Now let me get the Q3 expenses for all engineering team members: │
│     ├── Block 2                                                                  │
│     │   └── Tool Use: get_expenses                                               │
│     │       ├── ID: toolu_01Wu8LLTT2sKTTqpVwGT65Lj                               │
│     │       ├── Caller: model (direct)                                           │
│     │       └── Input:                                                           │
│     │           └── {                                                            │
│     │                 "employee_id": "ENG001",                                   │
│     │                 "quarter": "Q3"                                            │
│     │               }                                                            │
│     ├── Block 3                                                                  │
│     │   └── Tool Use: get_expenses                                               │
│     │       ├── ID: toolu_01KzjQ5mQJa9ocWjCGzYkD9F                               │
│     │       ├── Caller: model (direct)                                           │
│     │       └── Input:                                                           │
│     │           └── {                                                            │
│     │                 "employee_id": "ENG002",                                   │
│     │                 "quarter": "Q3"                                            │
│     │               }                                                            │
│     ├── Block 4                                                                  │
│     │   └── Tool Use: get_expenses                                               │
│     │       ├── ID: toolu_01RjjhZTg9JsKXE5E9S6Foho                               │
│     │       ├── Caller: model (direct)                                           │
│     │       └── Input:                                                           │
│     │           └── {                                                            │
│     │                 "employee_id": "ENG003",                                   │
│     │                 "quarter": "Q3"                                            │
│     │               }                                                            │
│     ├── Block 5                                                                  │
│     │   └── Tool Use: get_expenses                                               │
│     │       ├── ID: toolu_013xqpxpfc2N9rP5W5uMLAo9                               │
│     │       ├── Caller: model (direct)                                           │
│     │       └── Input:                                                           │
│     │           └── {                                                            │
│     │                 "employee_id": "ENG004",                                   │
│     │                 "quarter": "Q3"                                            │
│     │               }                                                            │
│     ├── Block 6                                                                  │
│     │   └── Tool Use: get_expenses                                               │
│     │       ├── ID: toolu_019zfzG6Wox8iDqy1dUXiH3t                               │
│     │       

╭────────────────────────────────────────────── Claude API Response ──────────────────────────────────────────────╮
│ Claude Message (assistant) │ tokens: 51,744 in • 290 out • 52,034 total                                         │
│ ├── Model: claude-sonnet-4-6                                                                           │
│ ├── Stop Reason: tool_use                                                                                       │
│ └── Content (7 blocks)                                                                                          │
│     ├── Block 1                                                                                                 │
│     │   └── Text                                                                                                │
│     │       └── Now let me calculate the approved travel expenses for each engineer and identify who exceeded   │
│     │           $5,000:                                                                                         │
│     ├── Block 2                                                                                                 │
│     │   └── Tool Use: get_custom_budget                                                                         │
│     │       ├── ID: toolu_013oegKwjvToLwEW1daDD8av                                                              │
│     │       ├── Caller: model (direct)                                                                          │
│     │       └── Input:                                                                                          │
│     │           └── {                                                                                           │
│     │                 "user_id": "ENG001"                                                                       │
│     │               }                                                                                           │
│     ├── Block 3                                                                                                 │
│     │   └── Tool Use: get_custom_budget                                                                         │
│     │       ├── ID: toolu_0162W4Ycr9FcVVED65exjAj4                                                              │
│     │       ├── Caller: model (direct)                                                                          │
│     │       └── Input:                                                                                          │
│     │           └── {                                                                                           │
│     │                 "user_id": "ENG003"                                                                       │
│     │               }                                                                                           │
│     ├── Block 4                                                                                                 │
│     │   └── Tool Use: get_custom_budget                                                                         │
│     │       ├── ID: toolu_01JcTX5rnwFxA99Am33gXmh6                                                              │
│     │       ├── Caller: model (direct)                                                                          │
│     │       └── Input:                                                                                          │
│     │           └── {                                                                                           │
│     │                 "user_id": "ENG005"                                                                       │
│     │               }                                                                                           │
│     ├── Block 5                                                                                                 │
│     │   └── Tool Use: get_custom_budget                        

╭────────────────────────────────────────────── Claude API Response ──────────────────────────────────────────────╮
│ Claude Message (assistant) │ tokens: 52,533 in • 992 out • 53,525 total                                         │
│ ├── Model: claude-sonnet-4-6                                                                           │
│ ├── Stop Reason: end_turn                                                                                       │
│ └── Content (1 blocks)                                                                                          │
│     └── Block 1                                                                                                 │
│         └── Text                                                                                                │
│             └── Now let me analyze the data. I'll calculate the approved travel expenses for each engineer:     │
│                                                                                                                 │
│                 **Analysis of Q3 Travel Expenses:**                                                             │
│                                                                                                                 │
│                 **ENG001 - Alice Chen (Senior Software Engineer)**                                              │
│                 - Approved travel expenses: $1,161.04 + $18.63 + $13.21 + $36.55 + $1,440.42 + $166.46 + $48.43 │
│                 + $1,124.56 + $1,245.90 + $1,498.42 = **$6,753.62**                                             │
│                 - Budget: $5,000 (Standard)                                                                     │
│                 - **EXCEEDED by $1,753.62** ❌                                                                  │
│                                                                                                                 │
│                 **ENG002 - Bob Martinez (Staff Engineer)**                                                      │
│                 - Approved travel expenses: $180.16 + $10.07 + $20.76 = **$210.99**                             │
│                 - Budget: $5,000 (Standard)                                                                     │
│                 - Under budget ✓                                                                                │
│                                                                                                                 │
│                 **ENG003 - Carol White (Software Engineer)**                                                    │
│                 - Approved travel expenses: $24.75 + $424.74 + $1,397.17 + $1,026.12 + $1,288.36 + $1,128.90 +  │
│                 $1,148.42 + $45.03 = **$6,483.49**                                                              │
│                 - Budget: $5,000 (Standard)                                                                     │
│                 - **EXCEEDED by $1,483.49** ❌                                                                  │
│                                                                                                                 │
│                 **ENG004 - David Kim (Principal Engineer)**                                                     │
│                 - Approved travel expenses: $21.68 + $46.12 + $1,008.68 + $46.43 = **$1,122.91**                │
│                 - Budget: $5,000 (Standard)                                                                     │
│                 - Under budget ✓                                                                                │
│                                                                                                                 │
│                 **ENG005 - Emma Johnson (Junior Software Engineer)                                              │
│                 ... (truncated)                                  

Result: Now let me analyze the data. I'll calculate the approved travel expenses for each engineer:

**Analysis of Q3 Travel Expenses:**

**ENG001 - Alice Chen (Senior Software Engineer)**
- Approved travel expenses: $1,161.04 + $18.63 + $13.21 + $36.55 + $1,440.42 + $166.46 + $48.43 + $1,124.56 + $1,245.90 + $1,498.42 = **$6,753.62**
- Budget: $5,000 (Standard)
- **EXCEEDED by $1,753.62** ❌

**ENG002 - Bob Martinez (Staff Engineer)**
- Approved travel expenses: $180.16 + $10.07 + $20.76 = **$210.99**
- Budget: $5,000 (Standard)
- Under budget ✓

**ENG003 - Carol White (Software Engineer)**
- Approved travel expenses: $24.75 + $424.74 + $1,397.17 + $1,026.12 + $1,288.36 + $1,128.90 + $1,148.42 + $45.03 = **$6,483.49**
- Budget: $5,000 (Standard)
- **EXCEEDED by $1,483.49** ❌

**ENG004 - David Kim (Principal Engineer)**
- Approved travel expenses: $21.68 + $46.12 + $1,008.68 + $46.43 = **$1,122.91**
- Budget: $5,000 (Standard)
- Under budget ✓

**ENG005 - Emma Johnson (Junior Software E

很好！可以看到 Claude 成功利用可用工具找出了超出差旅预算的团队成员。但同时也能看到，完成这项任务消耗了大量词元。Claude 必须通过它的上下文窗口消化全部费用明细——每位员工可能有 100 多条记录，每条都带有收据 URL、审批链、商户信息等大量元数据——然后才能解析它们、按类别汇总金额，并与预算限额比较。

此外，传统工具使用方式需要多次顺序往返：先获取团队成员，再获取每个人的费用，然后为超出标准限额的人检查自定义预算。每次往返都会增加时延，而且费用记录中丰富的元数据全部流经模型的上下文。

我们来看看能否用 PTC 提升性能：让 Claude 编写代码，改为在代码执行环境中处理这些大型数据集。

要为工具启用 PTC，我们必须先给任何应可通过代码执行调用的工具添加 `allowed_callers` 字段。

**需要考虑的要点**

- 未设置 allowed_callers 的工具默认只能由模型调用
- 通过指定多个调用方：`["direct", "code_execution_20250825"]`，工具可以同时由模型和代码执行调用
- 只对适合以编程方式/重复执行的工具启用。


In [7]:
import copy

ptc_tools = copy.deepcopy(tools)
for tool in ptc_tools:
    tool["allowed_callers"] = ["code_execution_20250825"]  # type: ignore


# Add the code execution tool
ptc_tools.append(
    {
        "type": "code_execution_20250825",  # type: ignore
        "name": "code_execution",
    }
)

现在我们已经更新工具定义以允许程序化工具使用，就可以用 PTC 运行智能体了。为此，我们对函数做了几处改动。我们必须使用 `beta` messages API。

1. 我们在 betas 中加入了 `"advanced-tool-use-2025-11-20"`。
2. 如果请求中定义了 `container_id`，我们把它随请求传入。只有对我们这样的有状态工作流才有必要；单轮工作流不需要。
3. 我们可以检查 `tool_use` 块中的 `caller` 字段，判断这次工具使用是来自模型的直接调用还是来自程序化调用。

注意，无论哪种情况，我们都通过 Claude API 发送工具结果，但只有 `direct` 调用会被模型“看到”。`code_execution_20250825` 类型的结果只有代码执行容器能看到。

In [8]:
messages = []


def run_agent_with_ptc(user_message):
    """Run agent using PTC"""
    messages.append({"role": "user", "content": user_message})
    total_tokens = 0
    start_time = time.time()
    container_id = None
    api_counter = 0

    while True:
        # Build request with PTC beta headers
        request_params = {
            "model": MODEL,
            "max_tokens": 4000,
            "tools": ptc_tools,
            "messages": messages,
        }

        response = client.beta.messages.create(
            **request_params,
            betas=[
                "advanced-tool-use-2025-11-20",
            ],
            extra_body={"container": container_id} if container_id else None,
        )
        viz.capture(response)
        api_counter += 1

        # Track container for stateful execution
        if hasattr(response, "container") and response.container:
            container_id = response.container.id
            print(f"\n[Container] ID: {container_id}")
            if hasattr(response.container, "expires_at"):
                # If the container has expired, we would need to restart our workflow. In our case, it completes before expiration.
                print(f"[Container] Expires at: {response.container.expires_at}")

        # Track token usage
        total_tokens += response.usage.input_tokens + response.usage.output_tokens

        if response.stop_reason == "end_turn":
            # Extract the first text block from the response
            final_response = next(
                (block.text for block in response.content if isinstance(block, BetaTextBlock)),
                None,
            )
            elapsed_time = time.time() - start_time
            return final_response, messages, total_tokens, elapsed_time, api_counter

        # As before, we process tool calls
        if response.stop_reason == "tool_use":
            # First, add the assistant's response to messages
            messages.append({"role": "assistant", "content": response.content})

            # Collect all tool results
            tool_results = []

            for block in response.content:
                if isinstance(block, BetaToolUseBlock):
                    tool_name = block.name
                    tool_input = block.input
                    tool_use_id = block.id

                    # We can use caller type to understand how the tool was invoked
                    caller_type = block.caller["type"]  # type: ignore

                    if caller_type == "code_execution_20250825":
                        print(f"[PTC] Tool called from code execution environment: {tool_name}")

                    elif caller_type == "direct":
                        print(f"[Direct] Tool called by model: {tool_name}")

                    result = tool_functions[tool_name](**tool_input)

                    # Format result as proper content for the API
                    if isinstance(result, list) and result and isinstance(result[0], str):
                        content = "\n".join(result)
                    elif isinstance(result, (dict, list)):
                        content = json.dumps(result)
                    else:
                        content = str(result)

                    tool_results.append(
                        {
                            "type": "tool_result",
                            "tool_use_id": tool_use_id,
                            "content": content,
                        }
                    )

            messages.append({"role": "user", "content": tool_results})

        else:
            print(f"\nUnexpected stop reason: {response.stop_reason}")
            elapsed_time = time.time() - start_time

            final_response = next(
                (block.text for block in response.content if isinstance(block, BetaTextBlock)),
                f"Stopped with reason: {response.stop_reason}",
            )
            return final_response, messages, total_tokens, elapsed_time, api_counter

In [9]:
# Run the PTC agent
result_ptc, conversation_ptc, total_tokens_ptc, elapsed_time_ptc, api_count_with_ptc = (
    run_agent_with_ptc(query)
)

╭────────────────────────────────────────────── Claude API Response ──────────────────────────────────────────────╮
│ Claude Message (assistant) │ tokens: 4,134 in • 539 out • 4,673 total                                           │
│ ├── Model: claude-sonnet-4-6                                                                           │
│ ├── Stop Reason: tool_use                                                                                       │
│ └── Content (3 blocks)                                                                                          │
│     ├── Block 1                                                                                                 │
│     │   └── Text                                                                                                │
│     │       └── I'll help you identify which engineering team members exceeded their Q3 travel budget. Let me   │
│     │           start by getting the engineering team members and their expenses.                               │
│     ├── Block 2                                                                                                 │
│     │   └── Server Tool Use                                                                                     │
│     │       ├── ID: srvtoolu_015mWPqaFni4B313UieCxbny                                                           │
│     │       ├── Caller: direct                                                                                  │
│     │       └── Code:                                                                                           │
│     │           └──    1                                                                                        │
│     │                  2 import asyncio                                                                         │
│     │                  3 import json                                                                            │
│     │                  4                                                                                        │
│     │                  5 async def main():                                                                      │
│     │                  6     # First, get all engineering team members                                          │
│     │                  7     team_members_json = await get_team_members({'department': 'engineering'})          │
│     │                  8     team_members = json.loads(team_members_json)                                       │
│     │                  9                                                                                        │
│     │                 10     print(f"Found {len(team_members)} engineering team members")                       │
│     │                 11                                                                                        │
│     │                 12     # Get Q3 expenses for all team members in parallel                                 │
│     │                 13     expense_tasks = [                                                                  │
│     │                 14         get_expenses({'employee_id': member['id'], 'quarter': 'Q3'})                   │
│     │                 15         for member in team_members                                                     │
│     │                 16     ]                                                                                  │
│     │                 17                                                                                        │
│     │                 18     expenses_results = await asyncio.gather(*expense_tasks)                            │
│     │                 19                                                                                        │
│     │                 20     # Calculate travel expenses for each member                                        │
│     │                 21     travel_spending = {}              


[Container] ID: container_011CVSAwq5J4vNPi3A4P2Rwh
[Container] Expires at: 2025-11-24 05:41:17.467494+00:00
[PTC] Tool called from code execution environment: get_team_members


╭──────────────────── Claude API Response ────────────────────╮
│ Claude Message (assistant) │ tokens: 0 in • 0 out • 0 total │
│ ├── Model: claude-sonnet-4-6                       │
│ ├── Stop Reason: tool_use                                   │
│ └── Content (8 blocks)                                      │
│     ├── Block 1                                             │
│     │   └── Tool Use: get_expenses                          │
│     │       ├── ID: toolu_01Nq2au3W69RmDFZaSdqe6u1          │
│     │       ├── Caller: code execution environment          │
│     │       └── Input:                                      │
│     │           └── {                                       │
│     │                 "employee_id": "ENG007",              │
│     │                 "quarter": "Q3"                       │
│     │               }                                       │
│     ├── Block 2                                             │
│     │   └── Tool Use: get_expenses                          │
│     │       ├── ID: toolu_01YYw9cuTSXk7bu7P38qBz6P          │
│     │       ├── Caller: code execution environment          │
│     │       └── Input:                                      │
│     │           └── {                                       │
│     │                 "employee_id": "ENG005",              │
│     │                 "quarter": "Q3"                       │
│     │               }                                       │
│     ├── Block 3                                             │
│     │   └── Tool Use: get_expenses                          │
│     │       ├── ID: toolu_01Fyxxe2KmVpVmw4jJL2CXSz          │
│     │       ├── Caller: code execution environment          │
│     │       └── Input:                                      │
│     │           └── {                                       │
│     │                 "employee_id": "ENG008",              │
│     │                 "quarter": "Q3"                       │
│     │               }                                       │
│     ├── Block 4                                             │
│     │   └── Tool Use: get_expenses                          │
│     │       ├── ID: toolu_01J4ovDu2UJa9Se19vxKTa6y          │
│     │       ├── Caller: code execution environment          │
│     │       └── Input:                                      │
│     │           └── {                                       │
│     │                 "employee_id": "ENG006",              │
│     │                 "quarter": "Q3"                       │
│     │               }                                       │
│     ├── Block 5                                             │
│     │   └── Tool Use: get_expenses                          │
│     │       ├── ID: toolu_01T24CrvQYA3LqGfZftCmueC          │
│     │       ├── Caller: code execution environment          │
│     │       └── Input:                                      │
│     │           └── {                                       │
│     │                 "employee_id": "ENG003",              │
│     │                 "quarter": "Q3"                       │
│     │               }                                       │
│     ├── Block 6                                             │
│     │   └── Tool Use: get_expenses                          │
│     │       ├── ID: toolu_01HotYZN6sk3gMLkpdWXbdz4          │
│     │       ├── Caller: code execution environment          │
│     │       └── Input:                                      │
│     │           └── {                                       │
│     │                 "employee_id": "ENG004",              │
│     │                 "quarter": "Q3"                       │
│     │               }                                       │
│     ├── Block 7                                             │
│     │   └── Tool Use: get_expenses                          │
│     │       ├── ID: toolu_01AxvEqi3AKqdnH44kGH1U6E          │
│     │       ├── Caller: code execution 


[Container] ID: container_011CVSAwq5J4vNPi3A4P2Rwh
[Container] Expires at: 2025-11-24 05:41:19.266670+00:00
[PTC] Tool called from code execution environment: get_expenses
[PTC] Tool called from code execution environment: get_expenses
[PTC] Tool called from code execution environment: get_expenses
[PTC] Tool called from code execution environment: get_expenses
[PTC] Tool called from code execution environment: get_expenses
[PTC] Tool called from code execution environment: get_expenses
[PTC] Tool called from code execution environment: get_expenses
[PTC] Tool called from code execution environment: get_expenses


╭─────────────────────────────────────── Claude API Response ───────────────────────────────────────╮
│ Claude Message (assistant) │ tokens: 4,751 in • 679 out • 5,430 total                             │
│ ├── Model: claude-sonnet-4-6                                                             │
│ ├── Stop Reason: tool_use                                                                         │
│ └── Content (6 blocks)                                                                            │
│     ├── Block 1                                                                                   │
│     │   └── Code Execution Result: Success (exit 0)                                               │
│     │       └── stdout:                                                                           │
│     │           └── Found 8 engineering team members                                              │
│     │                                                                                             │
│     │               Employees who exceeded $5,000 standard budget: 3                              │
│     │               ENG001: Alice Chen - $9177.88                                                 │
│     │               ENG003: Carol White - $6483.49                                                │
│     │               ENG007: Grace Taylor - $5289.35                                               │
│     │                                                                                             │
│     ├── Block 2                                                                                   │
│     │   └── Text                                                                                  │
│     │       └── Now let me check if any of these three employees have custom budget exceptions:   │
│     ├── Block 3                                                                                   │
│     │   └── Server Tool Use                                                                       │
│     │       ├── ID: srvtoolu_015GTpFmCbd2JPAQLAioB4Qb                                             │
│     │       ├── Caller: direct                                                                    │
│     │       └── Code:                                                                             │
│     │           └──    1                                                                          │
│     │                  2 import asyncio                                                           │
│     │                  3 import json                                                              │
│     │                  4                                                                          │
│     │                  5 async def main():                                                        │
│     │                  6     # Check custom budgets for the three employees who exceeded standard │
│     │                  7     exceeded_ids = ['ENG001', 'ENG003', 'ENG007']                        │
│     │                  8     exceeded_amounts = {                                                 │
│     │                  9         'ENG001': {'name': 'Alice Chen', 'travel_total': 9177.88},       │
│     │                 10         'ENG003': {'name': 'Carol White', 'travel_total': 6483.49},      │
│     │                 11         'ENG007': {'name': 'Grace Taylor', 'travel_total': 5289.35}      │
│     │                 12     }                                                                    │
│     │                 13                                                                          │
│     │                 14     # Get custom budgets in parallel                                     │
│     │                 15     budget_tasks = [                                                     │
│     │                 16         get_custom_budget({'user_id': emp_id})                           │
│     │                 17     


[Container] ID: container_011CVSAwq5J4vNPi3A4P2Rwh
[Container] Expires at: 2025-11-24 05:41:33.430636+00:00
[PTC] Tool called from code execution environment: get_custom_budget
[PTC] Tool called from code execution environment: get_custom_budget
[PTC] Tool called from code execution environment: get_custom_budget


╭────────────────────────────────────────────── Claude API Response ──────────────────────────────────────────────╮
│ Claude Message (assistant) │ tokens: 5,611 in • 205 out • 5,816 total                                           │
│ ├── Model: claude-sonnet-4-6                                                                           │
│ ├── Stop Reason: end_turn                                                                                       │
│ └── Content (2 blocks)                                                                                          │
│     ├── Block 1                                                                                                 │
│     │   └── Code Execution Result: Success (exit 0)                                                             │
│     │       └── stdout:                                                                                         │
│     │           └── ENGINEERING TEAM MEMBERS WHO EXCEEDED THEIR Q3 TRAVEL BUDGET:                               │
│     │               ================================================================================            │
│     │                                                                                                           │
│     │               Alice Chen (ENG001)                                                                         │
│     │                 Budget Limit: $5,000.00 (Standard)                                                        │
│     │                 Travel Spending: $9,177.88                                                                │
│     │                 Exceeded By: $4,177.88                                                                    │
│     │                                                                                                           │
│     │               Carol White (ENG003)                                                                        │
│     │                 Budget Limit: $5,000.00 (Standard)                                                        │
│     │                 Travel Spending: $6,483.49                                                                │
│     │                 Exceeded By: $1,483.49                                                                    │
│     │                                                                                                           │
│     │               Grace Taylor (ENG007)                                                                       │
│     │                 Budget Limit: $5,000.00 (Standard)                                                        │
│     │                 Travel Spending: $5,289.35                                                                │
│     │                 Exceeded By: $289.35                                                                      │
│     │                                                                                                           │
│     └── Block 2                                                                                                 │
│         └── Text                                                                                                │
│             └── ## Summary                                                                                      │
│                                                                                                                 │
│                 **Three engineering team members exceeded their Q3 travel budget:**                             │
│                                                                                                                 │
│                 1. **Alice Chen (ENG001)**                                                                      │
│                    - Budget: $5,000 (Standard)                                                                  │
│                    - Spent: $9,177.88                          

In [10]:
print(f"\n{'=' * 60}")
print(f"Result: {result_ptc}")
print(f"\n{'=' * 60}")
print("Performance Metrics:")
print(
    f"  Total API calls to Claude: {len([m for m in conversation_ptc if m['role'] == 'assistant'])}"
)
print(f"  Total tokens used: {total_tokens_ptc:,}")
print(f"  Total time taken: {elapsed_time_ptc:.2f}s")


Result: ## Summary

**Three engineering team members exceeded their Q3 travel budget:**

1. **Alice Chen (ENG001)**
   - Budget: $5,000 (Standard)
   - Spent: $9,177.88
   - Over budget by: **$4,177.88**

2. **Carol White (ENG003)**
   - Budget: $5,000 (Standard)
   - Spent: $6,483.49
   - Over budget by: **$1,483.49**

3. **Grace Taylor (ENG007)**
   - Budget: $5,000 (Standard)
   - Spent: $5,289.35
   - Over budget by: **$289.35**

All three employees are on the standard $5,000 quarterly travel budget with no custom exceptions, so they all genuinely exceeded their allocated travel budget for Q3.

Performance Metrics:
  Total API calls to Claude: 3
  Total tokens used: 15,919
  Total time taken: 34.88s


## 性能对比

我们来对比传统工具使用与 PTC 的性能：

**关于 API 调用次数的说明：** 你可能注意到 PTC 在这个示例中需要更多 API 调用。这是因为 PTC 编写的是更结构化、更顺序、遵循最佳实践的代码——例如把费用获取步骤与预算检查步骤分开。传统工具使用有时可以在一轮中把多个操作合并，但代价是把所有原始数据都送进模型的上下文。PTC 带来的词元效率收益远远超过往返次数的小幅增加，尤其是在处理元数据丰富的大型数据集时。

In [11]:
import pandas as pd

# Create comparison dataframe
comparison_data = {
    "Metric": [
        "API Calls",
        "Total Tokens",
        "Elapsed Time (s)",
        "Token Reduction",
        "Time Reduction",
    ],
    "Traditional": [
        api_count_without_ptc,
        f"{total_tokens:,}",
        f"{elapsed_time:.2f}",
        "-",
        "-",
    ],
    "PTC": [
        api_count_with_ptc,
        f"{total_tokens_ptc:,}",
        f"{elapsed_time_ptc:.2f}",
        f"{((total_tokens - total_tokens_ptc) / total_tokens * 100):.1f}%",
        f"{((elapsed_time - elapsed_time_ptc) / elapsed_time * 100):.1f}%",
    ],
}

df = pd.DataFrame(comparison_data)
print(df.to_string(index=False))

          Metric Traditional    PTC
       API Calls           4      4
    Total Tokens     110,473 15,919
Elapsed Time (s)       35.38  34.88
 Token Reduction           -  85.6%
  Time Reduction           -   1.4%


## 关键要点

在这个示例中，PTC 通过三项核心能力展现出显著的性能提升：

### 1. 在大规模数据解析中保护上下文
这是我们工作流演示的首要收益。Claude 编写代码，在代码执行环境内获取并处理数百条费用明细。通过以编程方式处理这些数据，Claude 解析 JSON、按状态过滤、按类别汇总金额，并与预算限额比较——全程没有把原始费用数据和元数据送进模型的上下文窗口。这带来了**词元用量的显著下降**。

### 2. 顺序依赖优化
该 API 存在顺序依赖：`get_custom_budget(user_id)` 只有在分析费用、确认谁超出标准 $5,000 预算之后才能调用。在传统工具使用中，这需要多次往返——获取团队成员、获取每个人的费用、识别超支者，再逐一检查他们的自定义预算。使用 PTC 时，Claude 编写代码，在代码执行环境中编排整个工作流，在循环中发起程序化工具使用，并在多次调用之间维护状态。这把原本许多次顺序 API 往返变成了更少次数、编排更聪明的调用。

### 3. 在代码执行中承担计算逻辑
模型不必在脑中跟踪并汇总带有复杂元数据的数十条费用，Claude 把算术与聚合逻辑交给了 Python 代码。这减轻了模型的认知负担，保证了计算精确，并让无关元数据（如收据 URL 和商户位置）完全不进入模型的上下文。

---

## 何时使用 PTC

在以下情况下 PTC 收益最大：

- **处理需要过滤、解析或聚合的元数据丰富的大型数据集**（比如我们的费用分析，涉及收据 URL、审批链、商户详情等）
- **存在顺序依赖**，即某次工具使用依赖此前调用的结果（比如只为超出标准限额的员工检查自定义预算）
- **需要多次工具使用**，在相似实体之间按顺序或循环进行（为每位团队成员检查费用和预算）
- **计算逻辑**可以减少需要流经模型上下文的内容
- **工具是安全的**，可以在无人监督的情况下以编程方式/重复执行

## 结论

我们的团队费用分析展现了 PTC 的优势：**在处理元数据丰富的大型数据集时大幅降低上下文消耗**，以及**优化存在顺序依赖的工作流**。通过让 Claude 编写代码来编排工具使用并以编程方式处理结果，我们在保持准确性和洞察质量的同时实现了可观的词元节省。

对于涉及带丰富元数据的批量数据处理、带依赖的重复工具使用，或者原始工具输出会污染模型上下文的场景，PTC 尤其有价值。

## 下一步

尝试把这一模式改造到你自己的用例中：
- 带顺序查询的财务数据分析与报告
- 依赖初次扫描结果的多实体健康检查
- 带元数据的大文件处理（CSV、JSON、XML 解析）
- 带后续查询的数据库查询结果聚合
- 基于初步结果的条件逻辑驱动的批量 API 操作